# EX3 — Właściwości metody Newtona i metod pochodnych

## Cel ćwiczenia

W tym ćwiczeniu zbadać podstawowe metody iteracyjne do rozwiązywania równań nie liniowych oraz porównać ich szybkość zbieżności:

- **Metoda Newtona** — zbieżność kwadratowa $\rho = 2$ (wymaga obliczania pochodnej)
- **Metoda strun (sieczna)** — zbieżność superliniowa $\rho \approx 1.618$ (stosunek złotego podziału $\phi$)
- **Metoda Regula Falsi (fałszywego przecięcia)** — zbieżność liniowa $\rho = 1$

W ogólnym przypadku szybkość zbieżności opisujemy zależnością:

$$e_{i+1} \approx C \cdot e_i^{\rho}$$

gdzie $e_i$ to błąd na iteracji $i$, a $\rho$ to wykładnik zbieżności.

### Metoda Newtona

Podstawowa iteracja:

$$x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)}$$

Dla podwójnych pierwiastków metoda Newtona traci zbieżność kwadratową — warto ją modyfikować.

### Metoda strun (sieczna)

Odrzuca się jeden punkt na każdej iteracji, a pochodną przybliża sieczną:

$$x_{i+1} = x_i - f(x_i) \cdot \frac{x_i - x_{i-1}}{f(x_i) - f(x_{i-1})}$$

Nie wymaga obliczania pochodnej, ale zachowuje tylko jeden punkt.

### Metoda Regula Falsi

Obie końce nawiasu są zachowywane, co daje gwarancję zbieżności, ale spowalnia ją do $\rho = 1$.

---

**Porównanie metod:** Newton (najszybsza, ale wymaga $f'$) vs Sieczna (kompromis) vs Regula Falsi (naj wolniejsza, ale najbezpieczniejsza).


In [ ]:
import numpy as np
import warnings

def calculateNewtonSolution(f, df, x0, opts=None):
    if opts is None:
        opts = {}
    imax = opts.get('imax', 1000000)
    return_all = opts.get('return_all', False)
    TOL = opts.get('TOL', 1e-12)
    show_warnings = opts.get('warnings', True)
    if return_all:
        x = np.zeros(imax)
        i = 0
        x[0] = x0 - f(x0)/df(x0)
        err = np.abs(x[0] - x0)
        while err >= TOL and i < imax - 1:
            i += 1
            x[i] = x[i-1] - f(x[i-1])/df(x[i-1])
            err = np.abs(x[i] - x[i-1])
        if show_warnings and i >= imax - 1:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x[:i+1]
    else:
        x_new = x0 - f(x0)/df(x0)
        err = np.abs(x_new - x0)
        i = 0
        while err >= TOL and i < imax:
            x_old = x_new
            x_new = x_old - f(x_old)/df(x_old)
            err = np.abs(x_new - x_old)
            i += 1
        if show_warnings and i >= imax:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x_new

def calculateFalsePositionSolution(f, a, b, opts=None):
    if opts is None:
        opts = {}
    imax = opts.get('imax', 1000000)
    return_all = opts.get('return_all', False)
    TOL = opts.get('TOL', 1e-12)
    show_warnings = opts.get('warnings', True)
    if return_all:
        x = np.zeros(imax)
        fx1, fx2 = f(a), f(b)
        if fx1 * fx2 >= 0:
            raise ValueError('Błędny wybór nawiasu.')
        c = b - (fx2*(b-a))/(fx2-fx1)
        x[0] = c; fc = f(c)
        x1, x2, fx1, fx2 = a, b, fx1, fx2; i = 0; c_old = 0
        while abs(c - c_old) > TOL and i < imax - 1:
            if fc < 0: x1, fx1 = c, fc
            else: x2, fx2 = c, fc
            c_old = c; c = x1 - (fx1*(x1-x2))/(fx1-fx2)
            i += 1; x[i] = c; fc = f(c)
        if show_warnings and i >= imax - 1:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x[:i+1]
    else:
        fx1, fx2 = f(a), f(b)
        if fx1 * fx2 >= 0:
            raise ValueError('Błędny wybór nawiasu.')
        c = b - (fx2*(b-a))/(fx2-fx1)
        fc = f(c)
        x1, x2, fx1, fx2 = a, b, fx1, fx2; i = 0; c_old = 0
        while abs(c - c_old) > TOL and i < imax:
            if fc < 0: x1, fx1 = c, fc
            else: x2, fx2 = c, fc
            c_old = c; c = x1 - (fx1*(x1-x2))/(fx1-fx2)
            fc = f(c)
            i += 1
        if show_warnings and i >= imax:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return c

def calculateSecantSolution(f, x0, opts=None):
    if opts is None:
        opts = {}
    imax = opts.get('imax', 1000000)
    return_all = opts.get('return_all', False)
    TOL = opts.get('TOL', 1e-12)
    show_warnings = opts.get('warnings', True)
    if return_all:
        x = np.zeros(imax)
        x[0] = x0; x[1] = 1.01 * x0
        err = 2 * TOL
        i = 1
        while err > TOL and i < imax - 1:
            i += 1
            x[i] = (x[i-2]*f(x[i-1]) - x[i-1]*f(x[i-2])) / (f(x[i-1]) - f(x[i-2]))
            err = abs(x[i] - x[i-1])
        if show_warnings and i >= imax - 1:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x[:i+1]
    else:
        x0_sec = x0
        x1_sec = 1.01 * x0
        x_old = x1_sec
        err = 2 * TOL
        i = 0
        while err > TOL and i < imax:
            x_new = (x0_sec*f(x1_sec) - x1_sec*f(x0_sec)) / (f(x1_sec) - f(x0_sec))
            err = abs(x_new - x_old)
            x0_sec = x1_sec
            x1_sec = x_new
            x_old = x_new
            i += 1
        if show_warnings and i >= imax:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x_new

def calculateSecantSolution2Points(f, x0, x1, opts=None):
    if opts is None:
        opts = {}
    imax = opts.get('imax', 1000000)
    return_all = opts.get('return_all', False)
    TOL = opts.get('TOL', 1e-12)
    show_warnings = opts.get('warnings', True)
    if return_all:
        x = np.zeros(imax)
        x[0] = x0; x[1] = x1
        err = 2 * TOL
        i = 1
        while err > TOL and i < imax - 1:
            i += 1
            x[i] = (x[i-2]*f(x[i-1]) - x[i-1]*f(x[i-2])) / (f(x[i-1]) - f(x[i-2]))
            err = abs(x[i] - x[i-1])
        if show_warnings and i >= imax - 1:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x[:i+1]
    else:
        x_old = x1
        err = 2 * TOL
        i = 0
        while err > TOL and i < imax:
            x_new = (x0*f(x_old) - x_old*f(x0)) / (f(x_old) - f(x0))
            err = abs(x_new - x_old)
            x0 = x_old
            x_old = x_new
            i += 1
        if show_warnings and i >= imax:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x_new

In [ ]:
import matplotlib.pyplot as plt

f = lambda x: (x-1)*(x-2)*(x-5)
df = lambda x: (x-1)*(x-2) + (x-1)*(x-5) + (x-2)*(x-5)
true_root = 1
options = {'imax': 100, 'return_all': True, 'TOL': 1e-8, 'warnings': True}
a, b = 0, 1.5
x0 = 0.5

resultsNewton = calculateNewtonSolution(f, df, x0, options)
errorNewton = np.abs(resultsNewton - true_root)
resultsFP = calculateFalsePositionSolution(f, a, b, options)
errorFP = np.abs(resultsFP - true_root)
resultsSecant = calculateSecantSolution(f, x0, options)
errorSecant = np.abs(resultsSecant - true_root)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(errorNewton, 'ro', label='Newton', marker='o')
ax.plot(errorFP, 'bo', label='Regula falsi', marker='o')
ax.plot(errorSecant, 'go', label='Sieczna', marker='o')
ax.set_xlabel('Iteracja')
ax.set_ylabel('Błąd')
ax.grid(True)
ax.legend()
ax.set_title('Błąd w funkcji iteracji')

ax = axes[1]
ax.plot(errorNewton[:-1], errorNewton[1:], 'ro', label='Newton')
ax.plot(errorFP[:-1], errorFP[1:], 'bo', label='Regula falsi')
ax.plot(errorSecant[:-1], errorSecant[1:], 'go', label='Sieczna')
ax.set_xlabel('Błąd $e_{i-1}$')
ax.set_ylabel('Błąd $e_i$')
ax.grid(True)
ax.legend()
ax.set_title('Błąd $e_i$ vs $e_{i-1}$')
ax.set_xlim([0, 0.2])
ax.set_ylim([0, 0.12])
plt.tight_layout()
plt.show()

print(f"Liczba iteracji: Newton={len(resultsNewton)}, Regula falsi={len(resultsFP)}, Sieczna={len(resultsSecant)}")
print(f"Pierwiastek: {true_root}")
print(f"Wyniki: Newton={resultsNewton[-1]:.12f}, Regula falsi={resultsFP[-1]:.12f}, Sieczna={resultsSecant[-1]:.12f}")

## ĆWICZENIE 1 — Szukanie różnych punktów startowych

Powyższy eksperyment pokazuje klasyczne zachowanie metod:

1. **Newton** — błędy maleją bardzo szybko (linia pionowa w wykresie $e_i$ vs $e_{i-1}$ oznacza zbieżność kwadratową).
2. **Sieczna** — zbieżność pośrednia między kwadratową a liniową.
3. **Regula Falsi** — wolna zbieżność liniowa (jedna krawędź nawiasu "przykleja się").

### Zadanie

Spróbuj znaleźć punkty startowe $x_0$ $[a, b]$, w których metoda Newtona **konkuruje** z metodą siecznej (tzn. zbieżność Newtona nie jest znacząco szybsza), oraz punkty, w których metoda Newtona **konkuruje** z metodą Regula Falsi.

Wskazówki:
- Jeśli pochodna $f'(x_0)$ jest bardzo mała, krok Newtona może być bardzo duży.
- Sprawdź zachowanie blisko pierwiastków podwójnych lub zbliskich pierwiastków.
- W funkcji testowej $f(x)=(x-1)(x-2)(x-5)$ pierwiastki są w $x=1$, $x=2$, $x=5$.


In [ ]:
from scipy.optimize import root_scalar

f2 = lambda x: x**3 - 2*x + 2
options = {'imax': 100, 'return_all': True, 'TOL': 1e-8, 'warnings': True}

try:
    true_root2 = root_scalar(f2, bracket=[-10, 0]).root
except:
    true_root2 = -1.7692923542386

print(f"Prawdziwy pierwiastek: {true_root2:.12f}")
print(f"Sprawdzenie: f({true_root2:.6f}) = {f2(true_root2):.2e}")

x0, x1 = 1, 5
resultsSecant = calculateSecantSolution2Points(f2, x0, x1, options)
errorSecant = np.abs(resultsSecant - true_root2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.semilogy(errorSecant, 'ro', marker='o')
ax.set_xlabel('Iteracja')
ax.set_ylabel('Błąd (skala logarytmiczna)')
ax.grid(True)
ax.set_title(f"Metoda sieczna: x0={x0}, x1={x1}")

ax = axes[1]
ax.semilogy(errorSecant[:-1], errorSecant[1:], 'ro', marker='o')
ax.set_xlabel('Błąd $e_{i-1}$ (log)')
ax.set_ylabel('Błąd $e_i$ (log)')
ax.grid(True)
ax.set_title('Analiza zbieżności: $e_i$ vs $e_{i-1}$')

plt.tight_layout()
plt.show()

print(f"Liczba iteracji: {len(resultsSecant)}")
print(f"Wynik końcowy: {resultsSecant[-1]:.12f}")
print(f"Błąd końcowy: {errorSecant[-1]:.2e}")


## ĆWICZENIE 2 — Modyfikacja punktów startowych dla metody siecznej

Równanie $f(x) = x^3 - 2x + 2$ ma pierwiastek rzeczywisty w okolicach $x \approx -1.769$.

### Zadanie

Zmień punkty startowe $x_0$, $x_1$ w metodzie siecznej i zaobserwuj:

1. **Wpływ odległości startowej** — czy duże $x_0$, $x_1$ spowalniają zbieżność?
2. **Symetrię punktów** — czy $x_0$, $x_1$ po obu stronach pierwiastka działają lepiej?
3. **Współczynnik zbieżności** — w log-log wykresie $e_i$ vs $e_{i-1}$, nachylenie prostej powinno odpowiadać $\rho \approx 1.618$.

Wskazówki:
- Spróbuj $(x_0, x_1) = (1.5, 2)$ — blisko pierwiastka.
- Spróbuj $(x_0, x_1) = (100, 200)$ — daleko od pierwiastka.
- Spróbuj $(x_0, x_1) = (-2, -1)$ — nawias zawierający pierwiastek.
- Porównaj liczbę iteracji dla różnych konfiguracji.


## Uwagi o szybkości zbieżności metod iteracyjnych

### Wykładnik zbieżności $\rho$

$$e_{i+1} \approx C \cdot e_i^{\rho}$$

Przyjmując logarytm po obu stronach:

$$\log(e_{i+1}) \approx \log(C) + \rho \cdot \log(e_i)$$

Na wykresie log-log nachylenie prostej odpowiada wykładnikowi $\rho$.

### Porównanie metod

| Metoda | $\rho$ | Wymaga $f'$? | Gwarancja zbieżności |
|--------|---------|--------------|---------------------|
| Newton | 2.0 | Tak | Nie (lokalna) |
| Sieczna | $\frac{1+\sqrt{5}}{2} \approx 1.618$ | Nie | Nie (lokalna) |
| Regula Falsi | 1.0 | Nie | Tak (globalna) |

### Uwagi dodatkowe

- Metoda Newtona jest najszybsza asympotycznie, ale przy złym punkcie startowym może rozbiec.
- Metoda sieczna jest kompromisem: szybsza niż Regula Falsi, ale wolniejsza niż Newton.
- Stosunek $\rho$ dla metody siecznej wynika z faktu, że wykorzystujemy $f(x_{i-1})$ do przybliżenia $f'(x_i)$, co wprowadza błąd rzędu $e_{i-1}$ w mianowniku.


## Tabela funkcji $f(x) = (x-r_1)(x-r_2)...(x-r_n)$ do eksperymentowania

| Nazwa | $f(x)$ | Pierwiastki | Pochodna $f'(x)$ |
|-------|--------|-------------|-------------------|
| Testowa (EX3) | $(x-1)(x-2)(x-5)$ | $1, 2, 5$ | $(x-1)(x-2) + (x-1)(x-5) + (x-2)(x-5)$ |
| Kulikowskiej | $(x^2+1)(x-1)(x-2)$ | $1, 2$, $\pm i$ | pochodna analityczna lub numeryczna |
| Wielomianowa | $x^3 - 2x + 2$ | $\approx -1.769$ | $3x^2 - 2$ |
| Transcendentalna | $\cos(x) - x$ | $\approx 0.739$ | $-\sin(x) - 1$ |
| Zbliskie pierwiastki | $(x-1)(x-1.01)(x-1.02)$ | $1, 1.01, 1.02$ | numeryczna |

### Porady

1. Do badań zbieżności metod siecznej i Newtona, warto spróbować funkcji z podwójnym lub bliskim pierwiastkiem.
2. W metodzie Newtona, jeśli $f'(x_i) \approx 0$, metoda jest niestabilna.
3. Metoda Regula Falsi wymaga nawiasu $[a, b]$ z $f(a) \cdot f(b) < 0$.
4. Do weryfikacji wyników porównuj z `scipy.optimize` jako referencją.
